In [ ]:
# %% Deep learning - Section 19.187
#    Code challenge 32: varying number of channels
#
#    1) Start from code from video 19.184 (emnist dataset)
#    2) Soft-code the number of channels of the 2 convolutional layers
#    3) Run an experiment with [2,5,8] kernels for ea§ch layer independently
#    4) Reduce the number of epochs from 10 to 5 for faster computation
#    5) Plot results as a 3x3 matrix (train and test error rates)
#    6) Plot total number of channels against train and test error rates (also
#       compute correlations)

# This code pertains a deep learning course provided by Mike X. Cohen on Udemy:
#   > https://www.udemy.com/course/deeplearning_x
# The "base" code in this repository is adapted (with very minor modifications)
# from code developed by the course instructor (Mike X. Cohen), while the
# "exercises" and the "code challenges" contain more original solutions and
# creative input from my side. If you are interested in DL (and if you are
# reading this statement, chances are that you are), go check out the course, it
# is singularly good.


In [1]:
# %% Libraries and modules
import numpy                  as np
import matplotlib.pyplot      as plt
import torch
import torch.nn               as nn
import seaborn                as sns
import copy
import torch.nn.functional    as F
import pandas                 as pd
import scipy.stats            as stats
import sklearn.metrics        as skm
import time
import sys
import imageio.v2             as imageio
import torchvision
import torchvision.transforms as T

from torch.utils.data                 import DataLoader,TensorDataset,Dataset
from sklearn.model_selection          import train_test_split
from google.colab                     import files
from torchsummary                     import summary
from scipy.stats                      import zscore
from sklearn.decomposition            import PCA
from scipy.signal                     import convolve2d
from torchsummary                     import summary
from IPython                          import display
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')
plt.style.use('default')


In [ ]:
# %% Use GPU

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)


In [ ]:
# %% Data

# Load data
# See also : https://www.nist.gov/itl/products-and-services/emnist-dataset
data = torchvision.datasets.EMNIST(root='emnist',split='letters',download=True)


In [4]:
# %% Preprocess data

# Transform into 4D tensor for convolution layers, transform from int8 to float
images = data.data.view([124800,1,28,28]).float()

# Remove N/A (first class category; relabel to start at 0)
letter_categories = data.classes[1:]
labels = copy.deepcopy(data.targets)-1

# Normalise
images /= torch.max(images)


In [5]:
# %% Create train and test datasets

# Split data with scikitlearn (10% test data)
train_data,test_data,train_labels,test_labels = train_test_split(images,labels,test_size=0.1)

# Convert to PyTorch datasets
train_data = TensorDataset(train_data,train_labels)
test_data  = TensorDataset(test_data,test_labels)

# Convert into DataLoader objects
batch_size   = 32
train_loader = DataLoader(train_data,batch_size=batch_size,shuffle=True,drop_last=True)
test_loader  = DataLoader(test_data,batch_size=test_data.tensors[0].shape[0])


In [6]:
# %% Function to generate the model

def gen_model(img_size,conv_chan1=2,conv_chan2=2,k=(3,3),s=(1,1)):

    class mnist_CNN(nn.Module):
        def __init__(self):
            super().__init__()

            # Convolution layer 1
            # output size: s = ((img+2*p-k)/s) + 1
            # post-pooling: s/2
            self.conv1 = nn.Conv2d(1,conv_chan1,kernel_size=k,stride=s,padding=1)

            height = np.floor( (img_size+2*self.conv1.padding[0]-k[0])/s[0] ) + 1
            height = np.floor( height/2 )
            width  = np.floor( (img_size+2*self.conv1.padding[0]-k[1])/s[1] ) + 1
            width  = np.floor( width/2 )

            # Convolution layer 2
            self.conv2 = nn.Conv2d(conv_chan1,conv_chan2,kernel_size=k,stride=s,padding=1)

            height = np.floor( (height+2*self.conv2.padding[0]-k[0])/s[0] ) + 1
            height = np.floor( height/2 )
            width  = np.floor( (width+2*self.conv2.padding[0]-k[1])/s[1] ) + 1
            width  = np.floor( width/2 )

            # Fully connected layer
            height = int(height)
            width  = int(width)
            chans  = int(conv_chan2)
            size   = height * width * chans

            self.fc1 = nn.Linear(size,50)

            # Output layer
            self.output = nn.Linear(50,26)

        def forward(self,x):

            # Adapt to export
            # MaxPool and ReLu on convolution layer 1
            x = F.relu(self.conv1(x))
            x = F.avg_pool2d(x,(2,2))

            # MaxPool and ReLu on convolution layer 2
            x = F.relu(self.conv2(x))
            x = F.avg_pool2d(x,(2,2))

            # Vectorise for linear layer
            x = torch.flatten(x,start_dim=1)

            # Linear and output layers
            x = F.relu(self.fc1(x))
            x = self.output(x)

            return x

    # Create model instance
    CNN = mnist_CNN()

    # Loss function
    loss_fun = nn.CrossEntropyLoss()

    # Optimizer
    optimizer = torch.optim.Adam(CNN.parameters(),lr=0.001)

    return CNN,loss_fun,optimizer


In [ ]:
# %% Test the model on one batch

CNN,loss_fun,optimizer = gen_model(images.size()[3],conv_chan1=2,conv_chan2=2,k=(3,3),s=(1,1))

X,y  = next(iter(train_loader))
yHat = CNN(X)

# Check sizes of output and target variable
print()
print(yHat.shape), print()
print(y.shape), print()

# Check loss
loss = loss_fun(yHat,y)
print(loss)


In [8]:
# %% Function to train the model

def train_model(img_size,conv_chan1,conv_chan2,k,s):

    # Parameters, model instance, inizialise vars
    num_epochs = 5
    CNN,loss_fun,optimizer = gen_model( img_size,
                                        conv_chan1=conv_chan1,
                                        conv_chan2=conv_chan2,
                                        k=k,
                                        s=s )

    # Ship model to GPU
    CNN.to(device)

    train_losses = []
    test_losses  = []
    train_acc    = []
    test_acc     = []

    # Loop over epochs
    for epoch_i in range(num_epochs):

        # Loop over training batches
        batch_acc  = []
        batch_loss = []

        for X,y in train_loader:

            # Ship data to GPU
            X = X.to(device)
            y = y.to(device)

            # Forward propagation and loss
            yHat = CNN(X)
            loss = loss_fun(yHat,y)

            # Backpropagation
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Loss and error rate (1-accuracy) from this batch
            yHat = yHat.cpu()
            y    = y.cpu()

            batch_loss.append(loss.item())
            batch_acc.append( torch.mean( (torch.argmax(yHat,axis=1) != y).float() ).item() )

        train_losses.append( np.mean(batch_loss) )
        train_acc.append( 100*np.mean(batch_acc) )

        # Test accuracy
        CNN.eval()

        with torch.no_grad():
            X,y = next(iter(test_loader))

            # Ship to GPU (y for loss calculation)
            X = X.to(device)
            y = y.to(device)

            yHat = CNN(X)
            loss = loss_fun(yHat,y)

            yHat = yHat.cpu()
            y    = y.cpu()

            test_acc.append( 100*torch.mean( (torch.argmax(yHat,axis=1) != y).float() ).item() )
            test_losses.append(loss.item())

        CNN.train()

    return train_acc,test_acc,train_losses,test_losses,CNN


In [9]:
# %% Fit model

# Parameters and preallocation
img_size   = images.size()[3]
conv_chan1 = [11,14,17] # [2,5,8]
conv_chan2 = [11,14,17] # [2,5,8]
k          = (3,3)
s          = (1,1)

results    = np.zeros((len(conv_chan1),len(conv_chan2),2))
chans_num  = np.zeros((len(conv_chan1),len(conv_chan2)))
model_time = np.zeros((len(conv_chan1),len(conv_chan2)))

# Loop over channels; takes ~7 mins on GPU and ~39 on CPU
for i,num_i in enumerate(conv_chan1):
    for j,num_j in enumerate(conv_chan2):

        start = time.time()

        train_err,test_err,train_losses,test_losses,CNN = train_model( img_size,
                                                                       conv_chan1[i],
                                                                       conv_chan2[j],
                                                                       k,
                                                                       s )

        model_time[i,j] = time.time() - start

        results[i,j,0]  = train_err[-1]
        results[i,j,1]  = test_err[-1]
        chans_num[i,j]  = num_i + num_j


In [ ]:
# %% Plotting

phi = (1 + np.sqrt(5)) / 2
fig,ax = plt.subplots(1,2,figsize=(phi*6,6))

for i in range(2):

    h = ax[i].imshow(results[:,:,i],vmin=np.min(results),vmax=np.max(results),cmap='plasma')
    ax[i].set_xlabel('Channels in conv. layer 1')
    ax[i].set_ylabel('Channels in conv. layer 2')
    ax[i].set_xticks(range(j+1)) # j from previous cell
    ax[i].set_yticks(range(j+1))
    ax[i].set_xticklabels(conv_chan1)
    ax[i].set_yticklabels(conv_chan1)
    title = 'Train' if i==0 else 'Test'
    ax[i].set_title('Error rates %s'%title,fontweight='bold')

# Colorbar (common colorscaling for both plots)
axpos = ax[1].get_position()
cax = fig.add_axes([axpos.x1+.01,axpos.y0,.01,.57])
hh = fig.colorbar(h,cax=cax)
hh.set_label('Error rate (%)',rotation=270,labelpad=10)

plt.savefig('figure130_code_challenge_32.png')
plt.show()
files.download('figure130_code_challenge_32.png')


In [ ]:
# %% Plotting

# Correlations
corr_train = np.corrcoef(chans_num.flatten(),results[:,:,0].flatten())
corr_test  = np.corrcoef(chans_num.flatten(),results[:,:,1].flatten())

# Plot
phi = (1 + np.sqrt(5)) / 2
fig = plt.figure(figsize=(phi*5,5))

plt.plot(chans_num.flatten(),results[:,:,0].flatten(),'o',alpha=0.7,
         label=f'Train (r={corr_train[0,1]:.2f})')
plt.plot(chans_num.flatten(),results[:,:,1].flatten(),'s',alpha=0.7,
         label=f'Test (r={corr_test[0,1]:.2f})')

plt.legend()
plt.xlabel('Total number of convolution channels')
plt.ylabel('Error rate (%)')

plt.savefig('figure131_code_challenge_32.png')
plt.show()
files.download('figure131_code_challenge_32.png')


In [ ]:
# %% Exercise 1
#    The correlation between error rate and convolution channels looks pretty compelling. How far do you dare go?!?! Try
#    adding more channels. Does the error rate simply keep going down until it reaches zero? Or do you find a point of
#    "diminishing returns", meaning that adding more channels no longer improves performance.

# Tried with [11,14,17]; apparently the pattern revers


In [ ]:
# %% Exercise 2
#    It seems intuitive that models with more layers take longer to train. But if there's one thing you've learned about
#    deep learning, it's that intuition doesn't always get us very far. Thus: modify the code to track the training time
#    for each model. Store the results in a separate matrix, and make an image of those results. Do they look like what
#    you had expected?

# Curious, quite a weird pattern

# Plot
phi = (1 + np.sqrt(5)) / 2
fig, ax = plt.subplots(1, 1, figsize=(phi*5,5))

h = ax.imshow(model_time,vmin=np.min(model_time),vmax=np.max(model_time),cmap='plasma')
ax.set_xlabel('Channels in conv. layer 1')
ax.set_ylabel('Channels in conv. layer 2')
ax.set_xticks(range(len(conv_chan1)))
ax.set_yticks(range(len(conv_chan2)))
ax.set_xticklabels(conv_chan1)
ax.set_yticklabels(conv_chan2)
ax.set_title('Model Training Time',fontweight='bold')

# Text annotations
vmin_val = np.min(model_time)
vmax_val = np.max(model_time)
for r in range(model_time.shape[0]):
    for c in range(model_time.shape[1]):
        val = model_time[r,c]
        normalized_val = (val - vmin_val) / (vmax_val - vmin_val)
        text_color = 'white' if normalized_val < 0.6 else 'black'
        ax.text(c,r,f'{val:.1f} s',ha='center',va='center',color=text_color)

cax = fig.add_axes([ax.get_position().x1+.01,ax.get_position().y0,0.01,ax.get_position().height])
hh  = fig.colorbar(h,cax=cax)
hh.set_label('Training Time (seconds)',rotation=270,labelpad=10)

plt.savefig('figure134_code_challenge_32_extra2.png')
plt.show()
files.download('figure134_code_challenge_32_extra2.png')


In [ ]:
# %% Exercise 3
#    Are net.train() and net.eval() necessary here? Why or why not?

# In the current model I didn't include any batch normalisation or drop out
# regularisation, nor anything whose bahaviour might change depending on whether
# we are in the training or test loop; but that said, I guess it's easy just to
# keep it, it doesn't slow the code down and it's just there if you need it
